# Figures for the Report

This notebook reads the result CSVs and saves PNG figures to a `figures/` folder. Drop the PNGs into the report.

In [ ]:
!pip install matplotlib pandas

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
fair = pd.read_csv("fairness_summary.csv")
blind = pd.read_csv("blind_screening_summary.csv")
display(fair)
display(blind)

In [ ]:
# Figure 1: average absolute difference per signal, before vs after blind screening.
labels = fair["changed_signal"].tolist()
before = fair["average_absolute_difference"].tolist()
after = blind.set_index("changed_signal").loc[labels, "average_blind_absolute_difference"].tolist()

x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([i - width/2 for i in x], before, width, label="Before blind screening")
ax.bar([i + width/2 for i in x], after, width, label="After blind screening")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("Average absolute score difference")
ax.set_title("SBERT score sensitivity by demographic signal")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig_before_after_blind.png", dpi=200)
plt.show()

In [ ]:
# Figure 2: per-domain breakdown of absolute differences.
comp = pd.read_csv("fairness_comparison.csv")
if "domain" not in comp.columns:
    # The comparison file does not always include domain; derive from job_title.
    domain_map = {
        "Software Engineer": "Software Engineering",
        "Financial Analyst": "Finance",
        "Marketing Coordinator": "Marketing",
        "Clinical Data Analyst": "Healthcare",
        "Academic Advisor": "Education",
    }
    comp["domain"] = comp["job_title"].map(domain_map)

by_domain = (
    comp.groupby(["domain", "changed_signal"])["absolute_difference"]
    .mean()
    .unstack()
)
by_domain.plot(kind="bar", figsize=(7, 4))
plt.ylabel("Average absolute score difference")
plt.title("Score sensitivity by domain and signal")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("figures/fig_by_domain.png", dpi=200)
plt.show()

In [ ]:
# Figure 3: distribution of score differences for the name change.
name_diffs = comp.loc[comp["changed_signal"] == "name", "score_difference"]
plt.figure(figsize=(6, 4))
plt.hist(name_diffs, bins=15, edgecolor="black")
plt.axvline(0, color="red", linestyle="--")
plt.xlabel("Score difference (changed - original)")
plt.ylabel("Number of resume-job pairs")
plt.title("Distribution of SBERT score change when only the name is swapped")
plt.tight_layout()
plt.savefig("figures/fig_name_distribution.png", dpi=200)
plt.show()

In [ ]:
from google.colab import files
for f in ["fig_before_after_blind.png", "fig_by_domain.png", "fig_name_distribution.png"]:
    files.download(f"figures/{f}")